# exp/r2-dmix on Colab (T4): queue [gru_v8p2z1, gru_v8p2z2]
Runtime -> Change runtime type -> T4 GPU, then Run all. A failed run raises with the config name and stops the queue (no silent skip). After a disconnect, just Run all again: each config resumes from its `last.pt` on Drive (`--resume`; model, optimizer, RNG), and a config whose `train_log.csv` already reached the final epoch or has an `early_stop` row is skipped. Each run writes `best.pt` (best validation PESQ, inference weights only), `last.pt` (full state, every epoch, ~28 MB) and `train_log.csv`; console output is appended to `train_stdout.log` there.

Queue (2026-11-23), both continuing the submission gru_v8p2y (val 3.5077 with rms_norm; train64 4.07, gap 0.56): init from weights/gru_v8p2y.pt (weights only, fresh Adam lr 5e-5), same loss (0.2 cmag + 0.32 torch-pesq), seed 0, 200 epochs, patience 30.
- **gru_v8p2z1**: train-time rms_norm, matching inference level mode rms_norm: each training input scaled by g = 0.079130 / rms(noisy), clean target by the same g; gain augmentation off; remix as before (p_original 0.5, else random clean + random train noise at SNR U[-5, 20] dB, peak <= 0.99). Validation and train64 also use rms_norm (+ output peak cap 0.99), i.e. they match eval/main.py.
- **gru_v8p2z2**: gru_v8p2z1 + full dynamic mixing (Zeghidour & Grangier 2021): every sample = random train clean + noise of a random other train file (random circular shift), with probability 0.2 a second noise at U[-10, 0] dB relative to the first, total noise scaled to SNR U[-5, 20] dB, peak <= 0.99, then rms_norm.

Time (estimate, docs/exp_r2_dmix.md): the gru_v8p2y run on the T4 took a median 8.6 s training + 20.3 s validation/train64 per epoch (~29 s); the new data options cost a few ms per step. About 1.6 h per config for 200 epochs (less with an early stop), about 3.3 h for the queue, plus ~5 min setup per session.

Background (exp/r2-dpesq): Differentiable-PESQ fine-tuning of gru_v5b: loss = w_cmag cmag + w_mrstft MR-STFT + w_sisdr (-SI-SDR) + lambda_pesq * torch-pesq loss (audiolabs/torch-pesq 0.1.2 through `src/nrc/tpesq.py`, which replaces the torchaudio functions it imports with exact pure-torch equivalents). lambda_pesq = 0.32 is set so that the PESQ term's gradient norm at gru_v5b init matches that of v5b's regression loss. All variants: warm start weights/gru_v5b.pt, magnitude mask, Adam lr 5e-5 (constant), seed 0, 100 epochs, val PESQ (pesq 0.0.4) every epoch, early stop after 20 epochs without gain. Each epoch also logs `train64_pesq` (pesq 0.0.4 on a fixed 64-file train subset), `gap` = train64 - val and `tpesq_mos` (mean torch-pesq MOS of the training batches).

| config | cmag | MR-STFT | -SI-SDR | lambda_pesq |
|---|---|---|---|---|
| gru_v8p1 | 1.0 | 0.5 | 0.3 | x1 |
| gru_v8p2 | 0.2 | 0 | 0 | x1 |
| gru_v8p3 | 0 | 0 | 0 | x1 (PESQ only) |
| gru_v8p2x | 0.2 | 0 | 0 | x1 (init gru_v8p2, 200 ep, patience 30) |
| gru_v8p2y | 0.2 | 0 | 0 | x1 (init gru_v8p2x, 200 ep, patience 30, gain U[-40, +6] dB) |

Time (estimate, docs/exp_r2_dpesq.md): no per-step PESQ calls, so training is GPU-bound. About 0.15-0.3 s/step on the T4 (not measured on GPU; epoch 1 prints s/step), plus about 20 s per epoch for pesq 0.0.4 on validation (164) + train64 (64) on 2 CPUs. That is about 35-45 s/epoch, up to about 1-1.3 h per variant for 100 epochs, and up to about 4 h for the queue (less with early stops, patience 20).

Drive layout: dataset at `MyDrive/noise_reduction_challenge/{train,validation}/...`; checkpoints go to `MyDrive/nrc_ckpt/<config_stem>/`.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content
import os
!rm -rf /content/repo && git clone -b exp/r2-dmix --depth 1 https://github.com/pauljh4323/SGSYS_13thTeam_NoiseReductionChallenge.git /content/repo
%cd /content/repo
for w in ('weights/gru_v8p2y.pt',):  # init_from of gru_v8p2z1 / gru_v8p2z2
    assert os.path.isfile(w) and os.path.getsize(w) > 1_000_000, f'{w} missing (init_from)'
    print('init weights OK:', w, os.path.getsize(w), 'bytes')
print('CPUs:', os.cpu_count())

In [ ]:
!pip install -q pesq==0.0.4 pystoi soundfile pyyaml
# torch-pesq without its torchaudio dependency (src/nrc/tpesq.py provides the pieces it uses); see requirements-dpesq.txt
!pip install -q --no-deps -r requirements-dpesq.txt
import subprocess, sys
r = subprocess.run([sys.executable, '-c', 'import sys; sys.path.insert(0, "src"); from nrc.tpesq import pesq_loss; import torch; '
                    'l = pesq_loss(1.0, 16000); x = torch.randn(2, 48000) * 0.1; print("torch-pesq OK, MOS(x, x) =", l.mos(x, x).tolist())'],
                   capture_output=True, text=True)
print(r.stdout, r.stderr[-2000:])
assert r.returncode == 0, 'torch-pesq import / smoke failed'

In [ ]:
!mkdir -p /content/data && cp -r --update=none /content/drive/MyDrive/noise_reduction_challenge /content/data/

In [ ]:
import os, yaml, pandas as pd
from pathlib import Path
CONFIGS = ["configs/dl/gru_v8p2z1.yaml", "configs/dl/gru_v8p2z2.yaml"]  # run in this order
for cfg in CONFIGS:
    assert os.path.isfile(cfg), f"{cfg} missing (wrong branch?)"
DATA = '/content/data/noise_reduction_challenge'
CKPT_ROOT = Path('/content/drive/MyDrive/nrc_ckpt')
CKPT_ROOT.mkdir(parents=True, exist_ok=True)
for cfg in CONFIGS:
    lp = CKPT_ROOT / Path(cfg).stem / 'last.pt'
    print(Path(cfg).stem, 'resume point:', lp, 'found' if lp.is_file() else 'not found (fresh start from init_from)')
import subprocess, sys

def run_logged(cmd, log, name):
    """Run cmd, stream its output here and append it to log; raise if it exits non-zero (no silent `| tee`)."""
    print('$ ' + ' '.join(map(str, cmd)), flush=True)
    with open(log, 'a', encoding='utf-8') as f:
        proc = subprocess.Popen([str(c) for c in cmd], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in proc.stdout:
            print(line, end='', flush=True)
            f.write(line)
        rc = proc.wait()
    if rc != 0:
        raise RuntimeError(f'{name} failed with exit code {rc}; see {log}')

In [ ]:
def ckpt_dir(cfg): return CKPT_ROOT / Path(cfg).stem

def finished(cfg):
    log = ckpt_dir(cfg) / 'train_log.csv'
    epochs = yaml.safe_load(open(cfg))['train']['epochs']
    if not log.is_file() or os.path.getsize(log) == 0:
        return False
    df = pd.read_csv(log)
    if len(df) == 0:
        return False
    early = 'early_stop' in df and (pd.to_numeric(df.early_stop, errors='coerce') == 1).any()
    return bool(early) or df.epoch.max() >= epochs

for cfg in CONFIGS:
    d = ckpt_dir(cfg)
    if finished(cfg):
        print(f'skip {cfg}: final epoch reached or early-stopped')
        continue
    d.mkdir(parents=True, exist_ok=True)
    run_logged([sys.executable, 'scripts/train_dpesq.py', '--config', cfg, '--data-root', DATA, '--ckpt-dir', d,
                '--resume', '--device', 'auto'], d / 'train_stdout.log', Path(cfg).stem)

In [ ]:
for cfg in CONFIGS:
    log = ckpt_dir(cfg) / 'train_log.csv'
    if not log.is_file():
        print(f'{Path(cfg).stem}: no train_log.csv'); continue
    df = pd.read_csv(log).dropna(subset=['val_pesq'])
    if df.empty:
        print(f'{Path(cfg).stem}: no validation yet'); continue
    r = df.loc[df.val_pesq.idxmax()]
    print(f'{Path(cfg).stem}: best epoch {int(r.epoch)}, val PESQ {r.val_pesq:.4f}, train64 PESQ {r.train64_pesq:.4f} '
          f'(gap {r.gap:+.4f}), val SNR {r.val_snr:.2f} dB, tpesq MOS {r.tpesq_mos:.3f} '
          f'(last epoch {int(pd.read_csv(log).epoch.max())})')